# 04 — Fine-tuning with LoRA
## LLM Lie Detector | Hallucination Detection Pipeline

This notebook fine-tunes Llama 3.2 3B using LoRA (Low-Rank Adaptation) to classify
whether a given question-answer pair contains a hallucination.

### Goals
- Load and prepare the combined dataset for training
- Configure LoRA for parameter-efficient fine-tuning
- Train the model with experiment tracking via Weights & Biases
- Evaluate against a naive baseline using F1-score, precision, and recall

### Approach
Rather than fine-tuning all 3 billion parameters, we use LoRA to train a small 
adapter — the industry standard approach for efficient LLM fine-tuning.

In [1]:
import os
os.environ["PYTHONUTF8"] = "1"

In [2]:
from dotenv import load_dotenv
import os
import wandb

load_dotenv('../.env')
wandb.login()
print("W&B login successful.")

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from WANDB_API_KEY.
wandb: Currently logged in as: itstamimmirza (itstamimmirza-rwth-aachen-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


W&B login successful.


In [3]:
import torch
import pandas as pd
import numpy as np
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from peft import LoraConfig, get_peft_model, TaskType
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

print("All imports successful.")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"GPU: {torch.cuda.get_device_name(0)}")

All imports successful.
CUDA available: True
GPU: NVIDIA GeForce RTX 4080 Laptop GPU


In [4]:
# Load the combined dataset
df = pd.read_csv('../data/combined_dataset.csv')

print(f"Total samples: {len(df)}")
print(f"Label distribution:\n{df['label'].value_counts()}")
print(f"\nSample:")
print(df.head(3).to_string())

Total samples: 15918
Label distribution:
label
1    8328
0    7590
Name: count, dtype: int64

Sample:
                                                                                           question                                                 answer  label    source
0  What television adaptation contains a character who was also adapted into a 2005 superhero film?  Batman Begins has a spin-off TV series called Gotham.      1  halueval
1      What punk rock band using horror film imagery was an influence on the Swedish band Entombed?                                                Misfits      0  halueval
2                                           Which lasted longer, Korean War or Battle of Mindanao?                   The Battle of Mindanao lasted longer.      1  halueval


In [5]:
# Split dataset - 90% train, 10% validation
train_df, val_df = train_test_split(
    df, 
    test_size=0.1, 
    random_state=42,
    stratify=df['label']  # ensures balanced split
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print(f"Training samples: {len(train_df)}")
print(f"Validation samples: {len(val_df)}")
print(f"\nTrain label distribution:\n{train_df['label'].value_counts()}")
print(f"\nVal label distribution:\n{val_df['label'].value_counts()}")

Training samples: 14326
Validation samples: 1592

Train label distribution:
label
1    7495
0    6831
Name: count, dtype: int64

Val label distribution:
label
1    833
0    759
Name: count, dtype: int64


In [6]:
from huggingface_hub import login
login()

model_id = "meta-llama/Llama-3.2-3B-Instruct"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Tokenizer loaded.")

Tokenizer loaded.


In [7]:
def format_training_example(row):
    label = "HALLUCINATED" if row['label'] == 1 else "TRUTHFUL"
    return {
        "text": f"Question: {row['question']}\nAnswer: {row['answer']}\nVerdict: {label}"
    }

train_formatted = train_df.apply(format_training_example, axis=1, result_type='expand')
val_formatted = val_df.apply(format_training_example, axis=1, result_type='expand')

from datasets import Dataset
train_dataset = Dataset.from_pandas(train_formatted)
val_dataset = Dataset.from_pandas(val_formatted)

print(f"Train: {len(train_dataset)} examples")
print(f"Val: {len(val_dataset)} examples")
print(f"\nSample:")
print(train_dataset[0]['text'])

Train: 14326 examples
Val: 1592 examples

Sample:
Question: What can broomsticks be used for?
Answer: Broomsticks can be used for flying
Verdict: HALLUCINATED


In [8]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig

# Load model normally - no sequence classification head
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

model.config.use_cache = False
print(f"Model loaded.")
print(f"VRAM: {torch.cuda.memory_allocated(0) / 1e9:.1f} GB")

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

Model loaded.
VRAM: 6.4 GB


In [9]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=["q_proj", "v_proj"],
    bias="none"
)

sft_config = SFTConfig(
    output_dir='../outputs/checkpoints',
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=4,
    warmup_steps=100,
    weight_decay=0.01,
    logging_steps=50,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    bf16=True,
    fp16=False,
    report_to="wandb",
    run_name="llama-hallucination-detector-v3",
    max_length=256,
    dataset_text_field="text"
)

trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    peft_config=lora_config,
)

print("Trainer ready.")

W0427 14:27:51.181000 5592 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels


Adding EOS to train dataset:   0%|          | 0/14326 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/14326 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/1592 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/1592 [00:00<?, ? examples/s]

Trainer ready.


In [11]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'pad_token_id': 128009}.


Epoch,Training Loss,Validation Loss
1,1.728785,1.752806
2,1.684333,1.711369
3,1.668854,1.701282


TrainOutput(global_step=2688, training_loss=1.797101936170033, metrics={'train_runtime': 3259.141, 'train_samples_per_second': 13.187, 'train_steps_per_second': 0.825, 'total_flos': 4.240108086854861e+16, 'train_loss': 1.797101936170033})